## Import Packages

In [28]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from tqdm import tqdm
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

# Load a PDF file
loader = PyPDFLoader("/Users/apple/Desktop/stuff/Projects/production-rag-pipeline/data/the fear bubble ch-1.pdf")
pages = loader.load()

## Text Splitter

In [29]:
text_splitter = CharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap = 50,
)
chunked_text = text_splitter.split_documents(pages)

## Generating Embeddings and Storing then in FAISS

In [30]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# This embeds all chunks AND stores them correctly in one call
vector_store = FAISS.from_documents(chunked_text, embeddings)

## Retrieval QnA

In [31]:
# 5. Initialize Chain
qa =RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-3.5-turbo"),
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True # Option to return source documents
)


In [32]:
q='What is the main topic of this document?'
answer = qa.invoke(q)

In [33]:
display(answer['result'])

'The main topic of the document is the narrator reflecting on his past experiences as a soldier, his struggle to adjust to civilian life after leaving the Special Forces, and his inner conflict between his previous identity as a warrior and his current success in popular culture.'

## Text splitter

### Character Text Splitter

In [34]:
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [35]:
# Load a PDF file
loader = PyPDFLoader("/Users/apple/Desktop/stuff/Projects/production-rag-pipeline/data/the fear bubble ch-1.pdf")
pages = loader.load()

In [36]:
text_splitter = CharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap = 50,
    separator='',
    strip_whitespace=False
)
chunked_text = text_splitter.split_documents(pages)


In [37]:
print(len(chunked_text))

89


In [38]:
print(f"The average chunk length is: {round(sum([len(i.page_content) for i in chunked_text])/len(chunked_text),2)} characters")


The average chunk length is: 468.35 characters


In [39]:
# Check actual chunk sizes
chunk_lengths = [len(i.page_content) for i in chunked_text]
print(f"Min: {min(chunk_lengths)}, Max: {max(chunk_lengths)}, Count: {len(chunked_text)}")


# See what the separator looks like in a raw page
print(repr(pages[0].page_content[:500]))

Min: 84, Max: 500, Count: 89
'CHAPTER 1\nTAMING THE GHOST OF METwenty Years Later‘Light?’‘Cheers, buddy.’I took a few rapid, light puffs of my cigar and heard it crackle into lifebetween my fingers. The smoke that licked the back of my throat was richand smooth, almost spicy. I took a deeper draw and peered at its glowingend. You could taste that it was expensive. But was this really £400 worthof cigar? That would make it, what, five quid a puff? I settled more deeplyinto the leather club chair and drew again, this time luxur'


## Recursive Text Splitter

In [40]:
text_splitter1 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunked_text1 = text_splitter1.split_documents(pages)

In [41]:
chunked_text1

[Document(metadata={'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'Preview', 'creationdate': "D:20260319163523Z00'00'", 'moddate': "D:20260319163523Z00'00'", 'author': 'Rahul Arora', 'title': 'the fear bubble', 'source': '/Users/apple/Desktop/stuff/Projects/production-rag-pipeline/data/the fear bubble ch-1.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}, page_content='CHAPTER 1'),
 Document(metadata={'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'Preview', 'creationdate': "D:20260319163523Z00'00'", 'moddate': "D:20260319163523Z00'00'", 'author': 'Rahul Arora', 'title': 'the fear bubble', 'source': '/Users/apple/Desktop/stuff/Projects/production-rag-pipeline/data/the fear bubble ch-1.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}, page_content='TAMING THE GHOST OF METwenty Years Later‘Light?’‘Cheers, buddy.’I took a few rapid, light puffs of my cigar and heard it crackle into lifebetween my fingers. The

In [48]:
print(len(chunked_text1))

90


In [49]:
print(f"The average chunk length is: {round(sum([len(i.page_content) for i in chunked_text1])/len(chunked_text1),2)} characters")


The average chunk length is: 459.44 characters


### 3rd Strategy

In [ ]:
 pip install nltk 

In [4]:
import nltk
nltk.download('punkt_tab')



[nltk_data] Downloading package punkt_tab to /Users/apple/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [12]:
from nltk.tokenize import sent_tokenize

In [20]:
all_text = " ".join([i.page_content for i in pages])

chunked_text2 = sent_tokenize(all_text)

In [21]:
len(chunked_text2)

437

In [24]:
chunks = []
current_chunk = ""

for sentence in sent_tokenize(all_text):
    if len(current_chunk) + len(sentence) <= 500:
        current_chunk += " " + sentence
    else:
        if current_chunk:
            chunks.append(current_chunk.strip())
        current_chunk = sentence
chunked_text2 = chunks

In [25]:
lengths = [len(c) for c in chunked_text2]
print(f"Count: {len(chunked_text2)}")
print(f"Avg: {round(sum(lengths)/len(lengths), 2)} chars")
print(f"Min: {min(lengths)}, Max: {max(lengths)}")


Count: 87
Avg: 430.9 chars
Min: 205, Max: 678


In [26]:
chunked_text2

['CHAPTER 1\nTAMING THE GHOST OF METwenty Years Later‘Light?’‘Cheers, buddy.’I took a few rapid, light puffs of my cigar and heard it crackle into lifebetween my fingers. The smoke that licked the back of my throat was richand smooth, almost spicy. I took a deeper draw and peered at its glowingend. You could taste that it was expensive. But was this really £400 worthof cigar? That would make it, what, five quid a puff?',
 'I settled more deeplyinto the leather club chair and drew again, this time luxuriating in theexperience, allowing the smoke to slide out through my lips gradually andwreathe about my face in silky ribbons.',
 'Before it dissipated, I took a sip ofthe rare single malt whisky my new barrister friend Ivan had also boughtme, this time at the bargain price of £60 a shot.‘So, how are you enjoying your new life?’ he asked me, his accent ascut-glass as the tumbler in my hand.His wry expression told me he probably wasn’t expecting an answer.After all, wasn’t it obvious? To al